In [25]:

import numpy as np
import matplotlib
import os
from itertools import product

import matplotlib.pyplot as plt
matplotlib.rcParams.update(matplotlib.rcParamsDefault)

In [36]:

def build_tiled_boundaries(ctcf_configs, chrom_size, region_size):
    """Tiles multiple CTCF configurations across the chromosome."""
    num_regions = chrom_size // region_size
    boundaryStrengthsL = np.zeros(chrom_size, dtype=np.double)
    boundaryStrengthsR = np.zeros(chrom_size, dtype=np.double)
    boundary_coordinates = []

    for i in range(num_regions):
        region_start = i * region_size
        config = ctcf_configs[i % len(ctcf_configs)]

        abs_sites_L = region_start + config['sites_L']
        abs_sites_R = region_start + config['sites_R']

        boundaryStrengthsL[abs_sites_L] = config['probs_L']
        boundaryStrengthsR[abs_sites_R] = config['probs_R']

        boundary_coordinates.extend(abs_sites_L)
        boundary_coordinates.extend(abs_sites_R)

    return boundaryStrengthsL, boundaryStrengthsR, np.array(boundary_coordinates, dtype=int)


def build_tiled_monomer_types(sticky_configs, chrom_size, region_size):
    """Tiles sticky elements (e.g., enhancers/promoters) across the chromosome."""
    num_regions = chrom_size // region_size
    monomer_types = np.zeros(chrom_size, dtype=int)

    for i in range(num_regions):
        region_start = i * region_size
        config = sticky_configs[i % len(sticky_configs)]
        abs_sites = region_start + config['sites']
        monomer_types[abs_sites] = config['types']

    return monomer_types


def build_tiled_mm_permm(mm_configs, chrom_size, region_size, cohesins_per_mm, loading_width):
    """Tiles the targeted loading elements across the chromosome."""
    num_regions = chrom_size // region_size
    mmLoc = np.ones(chrom_size, dtype=np.double)


    loading_bias = cohesins_per_mm / loading_width
    print(f"loading bias: {loading_bias}")
    for i in range(num_regions):
        region_start = i * region_size
        config = mm_configs[i % len(mm_configs)]
        abs_sites = region_start + config['sites']

        for site in abs_sites:
            mmLoc[(site-loading_width//2):(site+loading_width//2)] = loading_bias

    return mmLoc, loading_bias # this fulfills the need for birthArray, but is not normalized


def test_sweep(test_idx, region_size, CTCF_L, CTCF_R, sticky_elts, birthArray):
    """
    Plots a 1D 'dummy map' of the simulation parameters for a specific region.
    """
    test_region_start = region_size * test_idx
    test_region_end = test_region_start + region_size

    # 1. Slice out the specific region from the full chromosome arrays
    x = np.arange(test_region_start, test_region_end)
    ctcf_l_slice = CTCF_L[test_region_start:test_region_end]
    ctcf_r_slice = CTCF_R[test_region_start:test_region_end]
    sticky_slice = sticky_elts[test_region_start:test_region_end]
    birth_slice = birthArray[test_region_start:test_region_end]
    birth_slice = birth_slice / sum(birth_slice)  # Normalize

    # 2. Set up a stacked plot sharing the same X-axis
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(f"Bias {np.max(birthArray)/np.min(birthArray)}", fontsize=14)

    # Track 1: Left-pointing CTCF (Stall Right for translocator)
    axes[0].stem(x, ctcf_l_slice, linefmt='b-', markerfmt='bo', basefmt='k-')
    axes[0].set_ylabel('CTCF Left\nProb')
    axes[0].set_ylim(0, 1.1)

    # Track 2: Right-pointing CTCF (Stall Left for translocator)
    axes[0].stem(x, ctcf_r_slice, linefmt='r-', markerfmt='ro', basefmt='k-')
    axes[0].set_ylabel('CTCF Right\nProb')
    axes[0].set_ylim(0, 1.1)

    # Track 3: Sticky Elements (Enhancers/Promoters)
    # Using fill_between creates a nice blocky look for continuous regions
    axes[1].fill_between(x, 0, sticky_slice, color='green', step="mid", alpha=0.7)
    axes[1].set_ylabel('CREs')
    axes[1].set_yticks([0, 1])
    axes[1].set_ylim(0, 1.2)

    # Track 4: Targeted Loading Bias (birthArray)
    axes[2].plot(x, birth_slice, color='purple', drawstyle='steps-mid')
    axes[2].fill_between(x, 0, birth_slice, color='purple', alpha=0.3, step="mid")
    axes[2].set_ylabel('Loading Bias\n(birthArray)')
    axes[2].set_ylim([0, 0.01])
    # We let matplotlib auto-scale the Y-axis here because if birthArray
    # was already normalized (birthArray / sum), the values will be tiny.
    axes[2].set_ylim(bottom=0)

    # Clean up the bottom axis
    axes[2].set_xlabel('Genomic Coordinate (Monomer Index)', fontsize=12)

    #plt.tight_layout()
    return fig


def generate_param_grid(param_dict):
    """Yields a dictionary for each combination in the sweep space."""
    keys, values = zip(*param_dict.items())
    for bundle in product(*values):
        yield dict(zip(keys, bundle))
        
save_base_folder = "/mnt/md1/varshini/Blood/sim_data_sweep2/"
chrom_size = 70000
region_size = 2000

num_chains = 1
chain = [(x * chrom_size, (x + 1) * chrom_size, 0) for x in range(num_chains)]

N_monomers = chain[-1][1]
pause_prob = 1 - 0.0025

translocator_initialization_steps = 10000
smcStepsPerBlock = 1
steps_per_block = 50
restartBondUpdaterEveryBlocks = 3000
save_every_x_blocks = 3000
total_saved_blocks = 300
smcBondDist = 0.5
smcBondWiggleDist = 0.2
volume_density = 0.3
GPU_choice = 0

ctcf_configs = [{
    'name': 'default',
    'sites_L': np.array([574, 694, 866, 1241, 1390, 1580, 1752, 1800]),
    'probs_L': np.array([0.6, 0.8 , 0.95, 0.1 , 0.6, 0.6, 0.8, 0.1 ]),
    'sites_R': np.array([200, 330, 724, 1425, 1433, 1604]),
    'probs_R': np.array([0.9, 0.3 , 0.95, 0.4 , 0.3, 0.4 ])
}]

sticky_configs = [{
    'name': 'default',
    'sites': np.array([250, 372, 540, 745, 775, 833, 961, 1202, 1330, 1640, 1722]),
    'types': np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
}]

# matchmaker locations
mm_configs = [{
    'name': 'default',
    'sites': np.array([456, 1054, 1507]),
}]


EP_interaction_energy = 3.0
interaction_matrix = np.array([
    [0.0, 0.0],
    [0.0, EP_interaction_energy]
])


simulation_sweep_space = {
    'processivity': [300],
    'separations': [240],
    'ctcf_boost_factor': [4],
    'longlived_fraction': [0],
    'longlived_boost_factor': [20],
    'dsb_boost_factor': [0],
    'llp': [2],
    'h': [0.4],
    'loading_b': [2],  # targeted loading bias
    'loading_width': [4, 8, 16, 32], # width of targeted loading in kb,
    'cohesins_per_mm':[64] # if none, that means no bias
}


if __name__ == '__main__':

    stall_L, stall_R, boundary_coords = build_tiled_boundaries(ctcf_configs, chrom_size, region_size)
    monomer_types = build_tiled_monomer_types(sticky_configs, chrom_size, region_size)

    positions_to_sample = np.arange(chrom_size)
    block_to_save_all = []
    savedir = "/mnt/md0/varshini/Analysis/Blood/figs_v2/supp11"
    for i, config in enumerate(generate_param_grid(simulation_sweep_space)):

        run_name = (f"sim_proc{config['processivity']}_sep{config['separations']}"
                    f"_ctcf{config['ctcf_boost_factor']}_b{config['loading_b']}"
                    f"_h{config['h']}_llp{config['llp']}_w{config['loading_width']}_v0")
        save_folder = os.path.join(save_base_folder, run_name)

        if not os.path.exists(save_folder):
            os.makedirs(save_folder)

        print(f"\n{'=' * 50}")
        print(f"Starting Run: {run_name}")
        print(f"Config: {config}")
        print(f"{'=' * 50}")


        if config['cohesins_per_mm'] is not None:
            print(config['cohesins_per_mm'] )
            loading_loc_biases, loading_bias = build_tiled_mm_permm(mm_configs, chrom_size, region_size,
                                                      config['cohesins_per_mm'], config['loading_width'])
        else:
            loading_loc_biases = None
            loading_bias = 1
        smc_params = {
            'processivity': config['processivity'],
            'separations': config['separations'],
            'ctcf_boost_factor': config['ctcf_boost_factor'],
            'longlived_fraction': config['longlived_fraction'],
            'longlived_boost_factor': config['longlived_boost_factor'],
            'dsb_boost_factor': config['dsb_boost_factor'],
            'loading_loc_biases': loading_loc_biases
        }


        if loading_loc_biases is None:
            loading_loc_biases = np.ones(chrom_size)
        fig = test_sweep(0, region_size, stall_L, stall_R, monomer_types, loading_loc_biases)
        fig.savefig(os.path.join(savedir, f"params_{i}.pdf"))
        plt.close()
        #fig;


Starting Run: sim_proc300_sep240_ctcf4_b2_h0.4_llp2_w4_v0
Config: {'processivity': 300, 'separations': 240, 'ctcf_boost_factor': 4, 'longlived_fraction': 0, 'longlived_boost_factor': 20, 'dsb_boost_factor': 0, 'llp': 2, 'h': 0.4, 'loading_b': 2, 'loading_width': 4, 'cohesins_per_mm': 64}
64
loading bias: 16.0

Starting Run: sim_proc300_sep240_ctcf4_b2_h0.4_llp2_w8_v0
Config: {'processivity': 300, 'separations': 240, 'ctcf_boost_factor': 4, 'longlived_fraction': 0, 'longlived_boost_factor': 20, 'dsb_boost_factor': 0, 'llp': 2, 'h': 0.4, 'loading_b': 2, 'loading_width': 8, 'cohesins_per_mm': 64}
64
loading bias: 8.0

Starting Run: sim_proc300_sep240_ctcf4_b2_h0.4_llp2_w16_v0
Config: {'processivity': 300, 'separations': 240, 'ctcf_boost_factor': 4, 'longlived_fraction': 0, 'longlived_boost_factor': 20, 'dsb_boost_factor': 0, 'llp': 2, 'h': 0.4, 'loading_b': 2, 'loading_width': 16, 'cohesins_per_mm': 64}
64
loading bias: 4.0

Starting Run: sim_proc300_sep240_ctcf4_b2_h0.4_llp2_w32_v0
Conf